In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('../Datasets/Reviews.csv')

# Fill any empty text just in case
df['Summary'] = df['Summary'].fillna('')
df['Text'] = df['Text'].fillna('')

# Combine Summary and Text into one column
df['combined_text'] = df['Summary'] + ' ' + df['Text']

print("Dataset loaded!")
print("Shape:", df.shape)
df[['ProductId', 'Summary', 'Text', 'combined_text']].head()

Dataset loaded!
Shape: (100000, 11)


,ProductId,Summary,Text,combined_text
0,B001E4KFG0,Good Quality Dog Food,I have bought several of the Vitality canned d...,Good Quality Dog Food I have bought several of...
1,B00813GRG4,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Not as Advertised Product arrived labeled as J...
2,B000LQOCH0,"""Delight"" says it all",This is a confection that has been around a fe...,"""Delight"" says it all This is a confection tha..."
3,B000UA0QIQ,Cough Medicine,If you are looking for the secret ingredient i...,Cough Medicine If you are looking for the secr...
4,B006K2ZZ7K,Great taffy,Great taffy at a great price. There was a wid...,Great taffy Great taffy at a great price. The...


In [3]:
# Since multiple users reviewed same product
# combine all reviews per product into one
product_reviews = df.groupby('ProductId')['combined_text'].apply(
    lambda x: ' '.join(x)
).reset_index()

print("Unique products:", len(product_reviews))
product_reviews.head()

Unique products: 12560


,ProductId,combined_text
0,2734888454,made in china My dogs loves this chicken but i...
1,B00002N8SM,Doesn't catch fruit flies I don't know how thi...
2,B00002NCJC,thirty bucks? Why is this $[...] when the same...
3,B00002Z754,WOW Make your own 'slickers' ! I just received...
4,B00005V3DC,Best herbal tea for digestion If you're new to...


In [4]:
# Convert text into numbers
# max_features limits vocabulary to top 5000 words (keeps it fast)
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'  # removes common words like "the", "is", "and"
)

tfidf_matrix = tfidf.fit_transform(product_reviews['combined_text'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (12560, 5000)


In [5]:
# Find how similar each product is to every other product
product_similarity = cosine_similarity(tfidf_matrix)

product_similarity_df = pd.DataFrame(
    product_similarity,
    index=product_reviews['ProductId'],
    columns=product_reviews['ProductId']
)

print("Similarity matrix created!")
print("Shape:", product_similarity_df.shape)

Similarity matrix created!
Shape: (12560, 12560)


In [6]:
def recommend_similar_products(product_id, num_recommendations=5):
    # Check if product exists
    if product_id not in product_similarity_df.index:
        print("Product not found!")
        return
    
    # Get similarity scores for this product
    similar_products = product_similarity_df[product_id]
    
    # Sort by similarity, exclude the product itself
    similar_products = similar_products.drop(product_id)
    top_products = similar_products.nlargest(num_recommendations)
    
    print(f"Top {num_recommendations} products similar to {product_id}:")
    for i, (pid, score) in enumerate(top_products.items(), 1):
        print(f"{i}. Product: {pid} | Similarity Score: {round(score, 4)}")

# Test with first product in dataset
sample_product = product_reviews['ProductId'].iloc[0]
recommend_similar_products(sample_product)

Top 5 products similar to 2734888454:
1. Product: B0021L8XSW | Similarity Score: 0.571
2. Product: B003C5THEK | Similarity Score: 0.4877
3. Product: B00141UC9I | Similarity Score: 0.4448
4. Product: B001AJ1ULS | Similarity Score: 0.4448
5. Product: B001CPOR2E | Similarity Score: 0.4425


In [7]:
# For CBF evaluation we check if highly similar products
# also have similar average ratings

# Get average score per product
avg_scores = df.groupby('ProductId')['Score'].mean()

# Get actual vs predicted for products that exist in both
actual_scores = []
predicted_scores = []

for product_id in product_reviews['ProductId'][:500]:  # sample 500 products
    similar = product_similarity_df[product_id].drop(product_id).nlargest(5)
    for sim_product in similar.index:
        if sim_product in avg_scores and product_id in avg_scores:
            actual_scores.append(avg_scores[product_id])
            predicted_scores.append(avg_scores[sim_product])

mae = mean_absolute_error(actual_scores, predicted_scores)
mse = mean_squared_error(actual_scores, predicted_scores)
r2 = r2_score(actual_scores, predicted_scores)

print(f"MAE  : {round(mae, 4)}")
print(f"MSE  : {round(mse, 4)}")
print(f"R²   : {round(r2, 4)}")

MAE  : 0.8255
MSE  : 1.447
R²   : -0.4534


## Observations
 - TF-IDF converts product reviews into numerical vectors
 - Cosine similarity finds products with similar review content
 - Combined Summary + Text gives richer product representation
 - CBF works independently of user behavior
 - Works well even for new users (no ratings needed)